In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r"C:\Users\Payal\OneDrive\Desktop\gw_dashboard\data\gw_events_raw.csv")
df.head()

,event_name,commonName,version,catalog.shortName,GPS,reference,jsonurl,mass_1_source,mass_1_source_lower,mass_1_source_upper,...,far_upper,far_unit,p_astro,p_astro_lower,p_astro_upper,p_astro_unit,final_mass_source,final_mass_source_lower,final_mass_source_upper,final_mass_source_unit
0,GW250207_115645-v1,GW250207_115645,1,O4_Discovery_Papers,1.422965e+09,/o4_eventdata_docs/,https://gwosc.org/eventapi/json/O4_Discovery_P...,35.2,-1.7,1.7,...,NaN,1/yr,NaN,NaN,NaN,NaN,62.7,-1.6,1.0,M_sun
1,GW250114_082203-v1,GW250114_082203,1,O4_Discovery_Papers,1.420878e+09,/o4_eventdata_docs/,https://gwosc.org/eventapi/json/O4_Discovery_P...,33.6,-0.8,1.2,...,NaN,NaN,NaN,NaN,NaN,NaN,62.7,-1.1,1.0,M_sun
2,GW241110_124123-v1,GW241110_124123,1,O4_Discovery_Papers,1.415278e+09,/o4_eventdata_docs/,https://gwosc.org/eventapi/json/O4_Discovery_P...,17.2,-4.4,5.0,...,NaN,1/yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,GW241011_233834-v1,GW241011_233834,1,O4_Discovery_Papers,1.412725e+09,/o4_eventdata_docs/,https://gwosc.org/eventapi/json/O4_Discovery_P...,19.6,-2.5,3.6,...,NaN,1/yr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,GW240925_005809-v1,GW240925_005809,1,O4_Discovery_Papers,1.411261e+09,/o4_eventdata_docs/,https://gwosc.org/eventapi/json/O4_Discovery_P...,9.0,-1.0,2.0,...,NaN,1/yr,NaN,NaN,NaN,NaN,15.3,-0.4,0.7,M_sun


In [4]:
#df.info()
df.isnull().sum()

event_name                            0
commonName                            0
version                               0
catalog.shortName                     0
GPS                                   0
reference                             0
jsonurl                               0
mass_1_source                        88
mass_1_source_lower                  88
mass_1_source_upper                  88
mass_1_source_unit                   89
mass_2_source                        88
mass_2_source_lower                  88
mass_2_source_upper                  88
mass_2_source_unit                   89
network_matched_filter_snr           19
network_matched_filter_snr_lower    272
network_matched_filter_snr_upper    272
network_matched_filter_snr_unit     370
luminosity_distance                  88
luminosity_distance_lower            88
luminosity_distance_upper            88
luminosity_distance_unit             89
chi_eff                              90
chi_eff_lower                        90


In [5]:
summary = pd.DataFrame({
    "nulls": df.isnull().sum(),
    "dtype": df.dtypes,
    "unique": df.nunique()
})

summary

,nulls,dtype,unique
event_name,0,str,370
commonName,0,str,265
version,0,int64,5
catalog.shortName,0,str,15
GPS,0,float64,287
reference,0,str,21
jsonurl,0,str,370
mass_1_source,88,float64,196
mass_1_source_lower,88,float64,120
mass_1_source_upper,88,float64,168


In [6]:
keep_cols = [
    'event_name',
    'GPS',                      # GPS time of the event
    'mass_1_source',            # Mass of first object (solar masses)
    'mass_2_source',            # Mass of second object (solar masses)
    'total_mass_source',        # Combined mass
    'chirp_mass_source',        # A derived mass parameter
    'luminosity_distance',      # Distance in megaparsecs
    'network_matched_filter_snr', # Detection signal strength
    'far',                      # False alarm rate (lower = more confident)
    'catalog.shortName',        # Which catalog run (GWTC-1, GWTC-2, etc.)
    'commonName',               # Human-readable name like GW150914
]

# Only keep columns that actually exist in your data
keep_cols = [c for c in keep_cols if c in df.columns]
df = df[keep_cols].copy()

# Rename for readability
df.rename(columns={
    'catalog.shortName': 'catalog',
    'network_matched_filter_snr': 'snr',
    'luminosity_distance': 'distance_mpc',
    'total_mass_source': 'total_mass',
    'chirp_mass_source': 'chirp_mass',
    'far': 'false_alarm_rate'
}, inplace=True)

In [7]:
!pip install astropy

In [8]:
from astropy.time import Time

In [9]:
# gps time to real time

from astropy.time import Time

def gps_to_utc(gps_val):
    try:
        t = Time(float(gps_val), format='gps', scale='utc')
        return t.to_datetime().strftime('%Y-%m-%d')
    except:
        return None

df['event_date'] = df['GPS'].apply(gps_to_utc)
df['event_year'] = pd.to_datetime(df['event_date'], errors='coerce').dt.year

In [10]:
NS_THRESHOLD = 3.0  # Solar masses — objects below this are neutron stars

def classify_event(row):
    """Classify merger type from component masses.
    Returns: 'BBH', 'BNS', 'NSBH', or 'Unknown'"""
    m1 = pd.to_numeric(row['mass_1_source'], errors='coerce')
    m2 = pd.to_numeric(row['mass_2_source'], errors='coerce')

    if pd.isna(m1) or pd.isna(m2):
        return 'Unknown'

    both_bh  = m1 >= NS_THRESHOLD and m2 >= NS_THRESHOLD
    both_ns  = m1 <  NS_THRESHOLD and m2 <  NS_THRESHOLD
    mixed    = not both_bh and not both_ns

    if both_bh:  return 'BBH'
    if both_ns:  return 'BNS'
    if mixed:    return 'NSBH'
    return 'Unknown'

df['event_type'] = df.apply(classify_event, axis=1)
df['event_type'].value_counts()

event_type
BBH        263
Unknown     88
NSBH        15
BNS          4
Name: count, dtype: int64

In [11]:
RUN_BOUNDARIES = [
    ('O1',  '2015-09-12', '2016-01-19'),
    ('O2',  '2016-11-30', '2017-08-25'),
    ('O3a', '2019-04-01', '2019-10-01'),
    ('O3b', '2019-11-01', '2020-03-27'),
    ('O4',  '2023-05-24', '2025-06-01'),
]

def get_observing_run(date_str):
    try:
        d = pd.Timestamp(date_str)
    except:
        return 'Unknown'

    for run_name, start, end in RUN_BOUNDARIES:
        if pd.Timestamp(start) <= d <= pd.Timestamp(end):
            return run_name

    return 'Between runs'

df['observing_run'] = df['event_date'].apply(get_observing_run)
df['observing_run'].value_counts()

observing_run
O4              136
O3a             136
O3b              48
O2               33
O1               14
Between runs      3
Name: count, dtype: int64

In [12]:
mass_bins   = [0, 10, 20, 40, 60, 100, 9999]
mass_labels = ['0–10', '10–20', '20–40', '40–60', '60–100', '100+']

dist_bins   = [0, 500, 1000, 2000, 5000, 99999]
dist_labels = ['0–500 Mpc', '500–1k', '1k–2k', '2k–5k', '5k+ Mpc']

df['mass_bin']     = pd.cut(df['total_mass'],   bins=mass_bins,   labels=mass_labels)
df['distance_bin'] = pd.cut(df['distance_mpc'], bins=dist_bins,   labels=dist_labels)

# Cast to string — Power BI reads categorical bins best as plain strings
df['mass_bin']     = df['mass_bin'].astype(str)
df['distance_bin'] = df['distance_bin'].astype(str)

df[['total_mass', 'mass_bin']].head(10)

,total_mass,mass_bin
0,65.9,60–100
1,65.8,60–100
2,NaN,NaN
3,NaN,NaN
4,16.1,10–20
5,47.1,40–60
6,92.0,60–100
7,NaN,NaN
8,74.3,60–100
9,39.9,20–40


In [14]:
# Mass ratio: how unequal are the two objects? (1.0 = equal masses)
df['mass_ratio'] = (
    df['mass_2_source'] / df['mass_1_source']
).clip(upper=1.0)  # Ensure ratio never exceeds 1 (m2 <= m1 by convention)

# Detection confidence flag
df['high_confidence'] = df['false_alarm_rate'] < 1e-6

# Force numeric types for all measure columns
numeric_cols = ['mass_1_source', 'mass_2_source', 'total_mass',
                'chirp_mass', 'distance_mpc', 'snr', 'false_alarm_rate']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Sort by date ascending, reset index
df = df.sort_values('event_date').reset_index(drop=True)

# Export
df.to_csv('../data/gw_events_clean.csv', index=False)
df.to_excel(r'../exports/gw_events_clean.xlsx', index=False)
print(f"Final dataset: {df.shape[0]} events, {df.shape[1]} columns")
df.info()

Final dataset: 370 events, 19 columns
<class 'pandas.DataFrame'>
RangeIndex: 370 entries, 0 to 369
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_name        370 non-null    str    
 1   GPS               370 non-null    float64
 2   mass_1_source     282 non-null    float64
 3   mass_2_source     282 non-null    float64
 4   total_mass        269 non-null    float64
 5   chirp_mass        281 non-null    float64
 6   distance_mpc      282 non-null    float64
 7   snr               351 non-null    float64
 8   false_alarm_rate  339 non-null    float64
 9   catalog           370 non-null    str    
 10  commonName        370 non-null    str    
 11  event_date        370 non-null    str    
 12  event_year        370 non-null    int32  
 13  event_type        370 non-null    str    
 14  observing_run     370 non-null    str    
 15  mass_bin          269 non-null    str    
 16  distance_bin     